# Phase 9 — External Dataset Validation (Reviewer Comment 4)
Validates EW-MOGA on the Wisconsin Breast Cancer Diagnostic dataset.
569 instances, 30 features, binary classification (malignant vs benign).
Run this notebook after Notebook 02 completes.

In [1]:
# Cell 1 — Imports
import pandas as pd
import numpy as np
import shap
import time
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
from itertools import combinations
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SMOTE
from scipy.stats import spearmanr
from deap import base, creator, tools
print('All imports successful')

e:\project1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful


In [2]:
# Cell 2 — Configuration
BASE     = Path('E:/project1/EW_MOGA_Project')
EXT_DATA = BASE / 'data' / 'data.csv'
RESULTS  = BASE / 'results'
RESULTS.mkdir(exist_ok=True)

N_FEATURES_EXT     = 30
POP_SIZE           = 100
N_GEN              = 20
CX_PROB            = 0.8
N_FOLDS            = 5
SEEDS              = [0, 1, 2]
N_PERTURB          = 5
B_BOOTSTRAP        = 2
N_ABLATION_SAMPLES = 10
TOP_K              = 5
MUT_PROB_EXT       = 1.0 / N_FEATURES_EXT

CHECKPOINT_FILE = RESULTS / 'external_validation_checkpoint.csv'

print('Configuration set')
print(f'External data: {EXT_DATA}')

Configuration set
External data: E:\project1\EW_MOGA_Project\data\data.csv


In [3]:
# Cell 3 — Load and prepare external dataset
df_ext = pd.read_csv(EXT_DATA)

# Drop id and empty column
df_ext = df_ext.drop(columns=['id', 'Unnamed: 32'], errors='ignore')

# Convert diagnosis M/B to 1/0
df_ext['diagnosis'] = df_ext['diagnosis'].map({'M': 1, 'B': 0})

# Separate features and labels
FEATURE_COLS_EXT = [c for c in df_ext.columns if c != 'diagnosis']
X_ext = df_ext[FEATURE_COLS_EXT].values.astype(float)
y_ext = df_ext['diagnosis'].values.astype(int)

print(f'External dataset shape : {X_ext.shape}')
print(f'Number of features     : {len(FEATURE_COLS_EXT)}')
print(f'Class 0 (Benign)       : {(y_ext==0).sum()} ({100*(y_ext==0).mean():.1f}%)')
print(f'Class 1 (Malignant)    : {(y_ext==1).sum()} ({100*(y_ext==1).mean():.1f}%)')
print(f'Missing values         : {np.isnan(X_ext).sum()}')
print(f'Feature columns        : {FEATURE_COLS_EXT[:5]} ...')

External dataset shape : (569, 30)
Number of features     : 30
Class 0 (Benign)       : 357 (62.7%)
Class 1 (Malignant)    : 212 (37.3%)
Missing values         : 0
Feature columns        : ['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean'] ...


In [4]:
# Cell 4 — Preprocessing function
def preprocess(X_tr, X_val, y_tr, random_state=42):
    scaler   = MinMaxScaler()
    X_tr_sc  = scaler.fit_transform(X_tr)
    X_val_sc = scaler.transform(X_val)
    sm = SMOTE(random_state=random_state)
    X_tr_sm, y_tr_sm = sm.fit_resample(X_tr_sc, y_tr)
    return X_tr_sm, X_val_sc, y_tr_sm

print('Preprocessing function defined')

Preprocessing function defined


In [5]:
# Cell 5 — All component functions (SCI, FID, LFS, E(S), CCC, PR)
_prev_ranking = [None]

def compute_sci(X_tr, y_tr, mask, seed=42):
    idx = [i for i,v in enumerate(mask) if v==1]
    if not idx: return 0.0
    rng = np.random.RandomState(seed)
    rankings = []
    for _ in range(B_BOOTSTRAP):
        boot_idx = rng.choice(len(X_tr), len(X_tr), replace=True)
        X_b, y_b = X_tr[boot_idx], y_tr[boot_idx]
        rf = RandomForestClassifier(n_estimators=30, random_state=seed, n_jobs=-1)
        rf.fit(X_b[:, idx], y_b)
        bg  = X_b[rng.choice(len(X_b), min(50,len(X_b)), replace=False)]
        exp = shap.TreeExplainer(rf, bg)
        sv  = exp.shap_values(X_b[:min(30,len(X_b)),:], check_additivity=False)
        if isinstance(sv, list): sv_arr = np.mean([np.abs(r) for r in sv], axis=0)
        else: sv_arr = np.abs(sv)
        if sv_arr.ndim == 3: sv_arr = sv_arr.mean(axis=2)
        rankings.append(sv_arr.mean(axis=0))
    rhos = []
    for i, j in combinations(range(len(rankings)), 2):
        rho, _ = spearmanr(rankings[i], rankings[j])
        rhos.append(rho if not np.isnan(rho) else 0.0)
    return float(np.mean(rhos)) if rhos else 0.0

def compute_fid(X_tr, y_tr, mask, seed=42):
    idx = [i for i,v in enumerate(mask) if v==1]
    if not idx: return 1.0
    rf = RandomForestClassifier(n_estimators=30, random_state=seed, n_jobs=-1)
    rf.fit(X_tr[:, idx], y_tr)
    rng = np.random.RandomState(seed)
    bg  = X_tr[rng.choice(len(X_tr), min(50,len(X_tr)), replace=False)]
    exp = shap.TreeExplainer(rf, bg)
    sv  = exp.shap_values(X_tr[:min(30,len(X_tr)),:], check_additivity=False)
    if isinstance(sv, list): sv_arr = np.mean([np.abs(r) for r in sv], axis=0)
    else: sv_arr = np.abs(sv)
    if sv_arr.ndim == 3: sv_arr = sv_arr.mean(axis=2)
    curr_rank = np.argsort(-sv_arr.mean(axis=0))
    k = min(TOP_K, len(idx))
    curr_top = set(curr_rank[:k])
    if _prev_ranking[0] is None:
        _prev_ranking[0] = curr_top
        return 0.0
    prev_top = _prev_ranking[0]
    _prev_ranking[0] = curr_top
    inter = len(curr_top & prev_top)
    union = len(curr_top | prev_top)
    return 1.0 - (inter/union) if union > 0 else 0.0

def compute_lfs(rf, X_val, idx, seed=42):
    if len(X_val) == 0: return 0.0
    rng = np.random.RandomState(seed)
    sample_idx = rng.choice(len(X_val), min(N_ABLATION_SAMPLES,len(X_val)), replace=False)
    X_s = X_val[sample_idx][:, idx]
    bg  = X_s[:min(20,len(X_s))]
    exp = shap.TreeExplainer(rf, bg)
    sv  = exp.shap_values(X_s, check_additivity=False)
    if isinstance(sv, list): sv_arr = np.mean([np.abs(r) for r in sv], axis=0)
    else: sv_arr = np.abs(sv)
    if sv_arr.ndim == 3: sv_arr = sv_arr.mean(axis=2)
    importance_order = np.argsort(-sv_arr.mean(axis=0))
    base_proba = rf.predict_proba(X_s)
    drops = []
    X_ablated = X_s.copy()
    for feat_idx in importance_order:
        X_ablated[:, feat_idx] = 0.0
        new_proba = rf.predict_proba(X_ablated)
        drops.append(np.mean(np.abs(base_proba - new_proba)))
    return float(np.clip(np.mean(drops)*10, 0, 1))

def compute_es(X_tr, y_tr, X_val, mask, rf, idx, seed=42):
    sci = compute_sci(X_tr, y_tr, mask, seed)
    fid = compute_fid(X_tr, y_tr, mask, seed)
    lfs = compute_lfs(rf, X_val, idx, seed)
    return float(np.mean([sci, 1.0-fid, lfs]))

def compute_ccc(X_tr, y_tr, X_val, idx, seed=42):
    import xgboost as xgb
    from sklearn.svm import SVC
    from sklearn.neural_network import MLPClassifier
    classifiers = {
        'RF' : RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1),
        'XGB': xgb.XGBClassifier(n_estimators=100, random_state=seed,
                                  eval_metric='logloss', verbosity=0),
        'SVM': SVC(kernel='rbf', probability=True, random_state=seed),
        'MLP': MLPClassifier(hidden_layer_sizes=(64,32), max_iter=300, random_state=seed),
    }
    rankings = []
    rng = np.random.RandomState(seed)
    for name, clf in classifiers.items():
        try:
            clf.fit(X_tr[:, idx], y_tr)
            if name in ['RF','XGB']: exp = shap.TreeExplainer(clf)
            else:
                bg = X_tr[:, idx][rng.choice(len(X_tr), min(50,len(X_tr)), replace=False)]
                exp = shap.KernelExplainer(clf.predict_proba, bg)
            sv = exp.shap_values(X_val[:min(30,len(X_val))][:, idx], check_additivity=False)
            if isinstance(sv, list): sv_arr = np.mean([np.abs(r) for r in sv], axis=0)
            else: sv_arr = np.abs(sv)
            if sv_arr.ndim == 3: sv_arr = sv_arr.mean(axis=2)
            rankings.append(sv_arr.mean(axis=0))
        except: pass
    if len(rankings) < 2: return 0.0
    rhos = [spearmanr(rankings[i], rankings[j])[0]
            for i,j in combinations(range(len(rankings)),2)]
    return float(np.mean([r if not np.isnan(r) else 0.0 for r in rhos]))

def compute_pr(X_tr, y_tr, idx, seed=42):
    rf_b = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1)
    rf_b.fit(X_tr[:, idx], y_tr)
    rng = np.random.RandomState(seed)
    bg  = X_tr[rng.choice(len(X_tr), min(50,len(X_tr)), replace=False)]
    exp = shap.TreeExplainer(rf_b, bg)
    sv  = exp.shap_values(X_tr[:min(30,len(X_tr)),:], check_additivity=False)
    if isinstance(sv, list): sv_arr = np.mean([np.abs(r) for r in sv], axis=0)
    else: sv_arr = np.abs(sv)
    if sv_arr.ndim == 3: sv_arr = sv_arr.mean(axis=2)
    base_rank = sv_arr.mean(axis=0)
    rhos = []
    for p in range(N_PERTURB):
        boot_idx = rng.choice(len(X_tr), len(X_tr), replace=True)
        X_b, y_b = X_tr[boot_idx], y_tr[boot_idx]
        rf_p = RandomForestClassifier(n_estimators=100, random_state=seed+p, n_jobs=-1)
        rf_p.fit(X_b[:, idx], y_b)
        bg_p  = X_b[rng.choice(len(X_b), min(50,len(X_b)), replace=False)]
        exp_p = shap.TreeExplainer(rf_p, bg_p)
        sv_p  = exp_p.shap_values(X_b[:min(30,len(X_b)),:], check_additivity=False)
        if isinstance(sv_p, list): sv_p_arr = np.mean([np.abs(r) for r in sv_p], axis=0)
        else: sv_p_arr = np.abs(sv_p)
        if sv_p_arr.ndim == 3: sv_p_arr = sv_p_arr.mean(axis=2)
        rho, _ = spearmanr(base_rank, sv_p_arr.mean(axis=0))
        rhos.append(rho if not np.isnan(rho) else 0.0)
    return float(np.mean(rhos))

print('All component functions defined')

All component functions defined


In [6]:
# Cell 6 — NSGA-II setup for 30 features
for attr in ['FitnessExt','IndExt']:
    if attr in dir(creator): delattr(creator, attr)

creator.create('FitnessExt', base.Fitness, weights=(1.0, 1.0))
creator.create('IndExt', list, fitness=creator.FitnessExt)

def make_ind_ext():
    ind = [np.random.randint(0,2) for _ in range(N_FEATURES_EXT)]
    if sum(ind) == 0: ind[np.random.randint(N_FEATURES_EXT)] = 1
    return creator.IndExt(ind)

def mut_flip_ext(individual, indpb):
    for i in range(len(individual)):
        if np.random.random() < indpb: individual[i] = 1 - individual[i]
    if sum(individual) == 0: individual[np.random.randint(N_FEATURES_EXT)] = 1
    return individual,

tb_ext = base.Toolbox()
tb_ext.register('individual', make_ind_ext)
tb_ext.register('population', tools.initRepeat, list, tb_ext.individual)
tb_ext.register('mate',   tools.cxTwoPoint)
tb_ext.register('mutate', mut_flip_ext, indpb=MUT_PROB_EXT)
tb_ext.register('select', tools.selNSGA2)

print('NSGA-II toolbox configured for 30 features')

NSGA-II toolbox configured for 30 features


In [7]:
# Cell 7 — EW-MOGA run function for external dataset
def run_ewmoga_ext(X_tr, y_tr, X_val, y_val, seed=42):
    np.random.seed(seed)
    _prev_ranking[0] = None

    def eval_ind(ind):
        idx = [i for i,v in enumerate(ind) if v==1]
        if not idx: idx = [0]
        rf = RandomForestClassifier(n_estimators=30, random_state=seed, n_jobs=-1)
        rf.fit(X_tr[:, idx], y_tr)
        acc = accuracy_score(y_val, rf.predict(X_val[:, idx]))
        es  = compute_es(X_tr, y_tr, X_val, list(ind), rf, idx, seed)
        return acc, es

    tb_ext.register('evaluate', eval_ind)
    pop  = tb_ext.population(n=POP_SIZE)
    fits = list(map(tb_ext.evaluate, pop))
    for ind, fit in zip(pop, fits): ind.fitness.values = fit
    pop = tb_ext.select(pop, len(pop))

    for gen in range(N_GEN):
        offspring = tools.selTournamentDCD(pop, len(pop))
        offspring = [tb_ext.clone(o) for o in offspring]
        for c1, c2 in zip(offspring[::2], offspring[1::2]):
            if np.random.random() < CX_PROB:
                tb_ext.mate(c1, c2)
                del c1.fitness.values, c2.fitness.values
        for mut in offspring:
            tb_ext.mutate(mut)
            if not mut.fitness.valid: del mut.fitness.values
        invalid = [ind for ind in offspring if not ind.fitness.valid]
        fits    = list(map(tb_ext.evaluate, invalid))
        for ind, fit in zip(invalid, fits): ind.fitness.values = fit
        pop = tb_ext.select(pop + offspring, POP_SIZE)

    pareto = tools.sortNondominated(pop, len(pop), first_front_only=True)[0]
    best   = max(pareto, key=lambda ind: sum(ind.fitness.values))
    idx    = [i for i,v in enumerate(best) if v==1]
    if not idx: idx = [0]

    rf_f = RandomForestClassifier(n_estimators=100, random_state=seed, n_jobs=-1)
    rf_f.fit(X_tr[:, idx], y_tr)
    acc  = accuracy_score(y_val, rf_f.predict(X_val[:, idx]))
    es   = best.fitness.values[1]
    feat_names = ', '.join([FEATURE_COLS_EXT[i] for i in idx])
    ccc  = compute_ccc(X_tr, y_tr, X_val, idx, seed)
    pr   = compute_pr(X_tr, y_tr, idx, seed)

    return {
        'accuracy'              : acc,
        'n_features'            : len(idx),
        'feature_names'         : feat_names,
        'in_loop_explainability': es,
        'audit_ccc'             : ccc,
        'audit_pr'              : pr,
    }

print('EW-MOGA external run function defined')

EW-MOGA external run function defined


In [8]:
# Cell 8 — Run all 15 experiments on external dataset
# Resumes automatically from checkpoint if interrupted

if CHECKPOINT_FILE.exists():
    df_existing = pd.read_csv(CHECKPOINT_FILE)
    results     = df_existing.to_dict('records')
    completed   = set(zip(df_existing['seed'], df_existing['fold']))
    print(f'Resuming from checkpoint — {len(results)} runs already done')
else:
    results   = []
    completed = set()
    print('Starting fresh — no checkpoint found')

total   = len(SEEDS) * N_FOLDS
run_num = len(results)

for seed in SEEDS:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_ext, y_ext)):

        if (seed, fold) in completed:
            print(f'[SKIP] seed={seed} fold={fold} already in checkpoint')
            continue

        run_num += 1
        print(f'[{run_num}/{total}] seed={seed} fold={fold}', end=' ... ')
        t0 = time.time()

        X_tr,  X_val  = X_ext[tr_idx],  X_ext[val_idx]
        y_tr,  y_val  = y_ext[tr_idx],  y_ext[val_idx]
        X_tr_pp, X_val_pp, y_tr_pp = preprocess(X_tr, X_val, y_tr, seed)

        res     = run_ewmoga_ext(X_tr_pp, y_tr_pp, X_val_pp, y_val, seed=seed)
        elapsed = time.time() - t0

        results.append({
            'method'                : 'EW_MOGA_External',
            'dataset'               : 'Wisconsin_Breast_Cancer',
            'seed'                  : seed,
            'fold'                  : fold,
            'accuracy'              : res['accuracy'],
            'n_features'            : res['n_features'],
            'runtime_sec'           : elapsed,
            'feature_names'         : res['feature_names'],
            'in_loop_explainability': res['in_loop_explainability'],
            'audit_ccc'             : res['audit_ccc'],
            'audit_pr'              : res['audit_pr'],
        })

        # Save checkpoint after every run
        pd.DataFrame(results).to_csv(CHECKPOINT_FILE, index=False)
        print(f'acc={res["accuracy"]:.4f} ccc={res["audit_ccc"]:.3f} '
              f'pr={res["audit_pr"]:.3f} [{elapsed:.0f}s] \u2713 saved')
        completed.add((seed, fold))

df_ext_results = pd.DataFrame(results)
df_ext_results.to_csv(RESULTS / 'external_validation_results.csv', index=False)
print(f'\nAll {len(df_ext_results)} runs complete')
print(f'Saved: results/external_validation_results.csv')

Starting fresh — no checkpoint found
[1/15] seed=0 fold=0 ... 

100%|██████████| 30/30 [00:01<00:00, 18.92it/s]


acc=0.9386 ccc=0.823 pr=0.984 [686s] ✓ saved
[2/15] seed=0 fold=1 ... 

100%|██████████| 30/30 [00:01<00:00, 18.26it/s]


acc=0.9912 ccc=0.655 pr=0.989 [689s] ✓ saved
[3/15] seed=0 fold=2 ... 

100%|██████████| 30/30 [00:00<00:00, 39.81it/s]


... (21 outputs left out)

e:\project1\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
e:\project1\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
e:\project1\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
e:\project1\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should b

acc=0.9825 ccc=0.762 pr=0.982 [970s] ✓ saved
[15/15] seed=2 fold=4 ... 

e:\project1\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
e:\project1\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
e:\project1\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
e:\project1\.venv\Lib\site-packages\sklearn\utils\parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should b

acc=0.9912 ccc=0.698 pr=0.989 [1185s] ✓ saved

All 15 runs complete
Saved: results/external_validation_results.csv


In [9]:
# Cell 9 — Summary of external validation results
df_ext_r = pd.read_csv(RESULTS / 'external_validation_results.csv')

print('='*60)
print('EXTERNAL VALIDATION SUMMARY')
print('Dataset: Wisconsin Breast Cancer (n=569, 30 features)')
print('='*60)
print(f'Accuracy  : {df_ext_r.accuracy.mean():.4f} ± {df_ext_r.accuracy.std():.4f}')
print(f'CCC       : {df_ext_r.audit_ccc.mean():.4f} ± {df_ext_r.audit_ccc.std():.4f}')
print(f'PR        : {df_ext_r.audit_pr.mean():.4f} ± {df_ext_r.audit_pr.std():.4f}')
print(f'N features: {df_ext_r.n_features.mean():.2f} ± {df_ext_r.n_features.std():.2f}')
print(f'Runtime   : {df_ext_r.runtime_sec.mean():.1f}s mean')
print()
print('Per run results:')
print(df_ext_r[['seed','fold','accuracy','audit_ccc','audit_pr']].to_string(index=False))

EXTERNAL VALIDATION SUMMARY
Dataset: Wisconsin Breast Cancer (n=569, 30 features)
Accuracy  : 0.9766 ± 0.0168
CCC       : 0.7026 ± 0.0897
PR        : 0.9872 ± 0.0070
N features: 12.33 ± 1.45
Runtime   : 752.7s mean

Per run results:
 seed  fold  accuracy  audit_ccc  audit_pr
    0     0  0.938596   0.823260  0.984344
    0     1  0.991228   0.655012  0.988996
    0     2  0.982456   0.761616  0.993807
    0     3  0.973684   0.788278  0.972529
    0     4  0.982301   0.573260  0.986627
    1     0  0.964912   0.674825  0.991378
    1     1  1.000000   0.626374  0.989562
    1     2  0.964912   0.854545  0.991773
    1     3  0.982456   0.644689  0.981843
    1     4  0.982301   0.680556  0.997834
    2     0  0.973684   0.604895  0.995349
    2     1  0.991228   0.592075  0.986614
    2     2  0.947368   0.799267  0.975570
    2     3  0.982456   0.761905  0.981861
    2     4  0.991150   0.697802  0.989236


In [10]:
# Cell 10 — Cross-dataset comparison table
orig_path = RESULTS / 'ew_moga_results.csv'
ext_path  = RESULTS / 'external_validation_results.csv'

if not orig_path.exists():
    print('ew_moga_results.csv not found — run Notebook 02 first')
else:
    df_orig  = pd.read_csv(orig_path)
    df_ext_r = pd.read_csv(ext_path)

    comparison = pd.DataFrame({
        'Metric': ['Accuracy','CCC','PR','Features Selected'],
        'Brain Tumor MRI (Original)': [
            f"{df_orig.accuracy.mean():.4f} \u00b1 {df_orig.accuracy.std():.4f}",
            f"{df_orig.audit_ccc.mean():.4f} \u00b1 {df_orig.audit_ccc.std():.4f}",
            f"{df_orig.audit_pr.mean():.4f} \u00b1 {df_orig.audit_pr.std():.4f}",
            f"{df_orig.n_features.mean():.2f} \u00b1 {df_orig.n_features.std():.2f}",
        ],
        'Wisconsin Breast Cancer (External)': [
            f"{df_ext_r.accuracy.mean():.4f} \u00b1 {df_ext_r.accuracy.std():.4f}",
            f"{df_ext_r.audit_ccc.mean():.4f} \u00b1 {df_ext_r.audit_ccc.std():.4f}",
            f"{df_ext_r.audit_pr.mean():.4f} \u00b1 {df_ext_r.audit_pr.std():.4f}",
            f"{df_ext_r.n_features.mean():.2f} \u00b1 {df_ext_r.n_features.std():.2f}",
        ],
    })

    print('='*65)
    print('CROSS-DATASET COMPARISON TABLE')
    print('='*65)
    print(comparison.to_string(index=False))
    print()
    comparison.to_csv(RESULTS / 'cross_dataset_comparison.csv', index=False)
    print('Saved: results/cross_dataset_comparison.csv')
    print()
    print('Notebook 9 complete')

CROSS-DATASET COMPARISON TABLE
           Metric Brain Tumor MRI (Original) Wisconsin Breast Cancer (External)
         Accuracy            0.9902 ± 0.0035                    0.9766 ± 0.0168
              CCC            0.6210 ± 0.1451                    0.7026 ± 0.0897
               PR            0.9651 ± 0.0324                    0.9872 ± 0.0070
Features Selected                5.73 ± 1.33                       12.33 ± 1.45

Saved: results/cross_dataset_comparison.csv

Notebook 9 complete
